# 04 - Bangla-BERT -> BERT2BERT

sagorsarker/bangla-bert-base turned into an encoder-decoder (BERT2BERT). **Naming:** this checkpoint is *Bangla-BERT-Base* by Sagor Sarker, which is a different model from *BanglaBERT* by csebuetnlp (Bhattacharjee et al., 2022). Name and cite it correctly in the paper.

The decoder's cross-attention layers start from **random weights**, so its zero-shot output is meaningless by design. Report it as *n/a* or explain this.

**Fixes vs. your failed run:** LR 2e-4 -> 5e-5 with 10% warmup, a tokenizer check (Bengali vowel signs must not be stripped), corrected label alignment, matching generation config, and WordPiece detokenisation.

**Run on Kaggle:** Settings > Accelerator = GPU (T4 or P100), Internet = ON, then **Save Version > Save & Run All (Commit)**.
Before the first run: **Add Input > Datasets** and add `chittagonian-splits` (the saved output of `00_data_prep`).
No other dataset is needed.
When it finishes, download `results_<model>.zip` from the **Output** tab. One model per run, each in its own Kaggle run.
Only **Cell 2 (CONFIG)** differs between model notebooks. Cells 3-10 are identical in all of them, which is what makes the comparison fair.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU fragmentation / OOM
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use ONE GPU even if Kaggle offers T4 x2 (same batch size / memory for every model)
# -- CELL 1: Environment & Dependencies ----------------------------------
# Same packages in every model notebook. (Streamlit / tunnel packages live in the demo notebook.)
!pip install -q transformers datasets evaluate accelerate
!pip install -q sacrebleu rouge_score bert_score sentencepiece protobuf


In [ ]:
# -- CELL 2: CONFIG - the ONLY cell that differs between model notebooks -
MODEL_NAME = "Bangla-BERT2BERT"
CFG = {
    "model_id":    "sagorsarker/bangla-bert-base",
    "type":        "bert2bert",
    "tie_weights": False,       # True = shared encoder/decoder weights (smaller, sometimes better)
    "eval_bs":     8,
    "color":       "#e67e22",
}
HP = {  # FIX: 2e-4 made training diverge (loss ~ ln(vocab) ~ 11 = random guessing).
        # BERT-based models are normally fine-tuned at 1e-5 - 5e-5 with warmup.
    "lr": 5e-5, "warmup_ratio": 0.1, "bs": 8, "ga": 1, "ckpt": False,
    "fp16": False, "epochs": 10, "patience": 3,
}


In [ ]:
# -- CELL 3: Shared Setup - Paths, Seeds, Frozen Splits, Metrics ---------
# [WARN] This cell is IDENTICAL in every model notebook (01-04). Do not edit it in
#   only one notebook, or the comparison stops being fair.
import os, re, json, gc, time, random, hashlib, shutil, warnings, platform
import numpy as np, pandas as pd, torch
import transformers, evaluate, sacrebleu
from sacrebleu.metrics import BLEU, CHRF
from transformers import set_seed, GenerationConfig
import glob
warnings.filterwarnings("ignore")

# Kaggle: outputs of THIS run go to /kaggle/working. The frozen splits are READ from the
# output of notebook 00 (added to this notebook as an input).
project_dir = '/kaggle/working/Chittagonian_AI_Project'
results_dir = f'{project_dir}/results_v5'
os.makedirs(results_dir, exist_ok=True)
SPLITS_DATASET = '/kaggle/input/datasets/falaqueabrar69/chittagonian-splits'   # your dataset = saved output of 00_data_prep
data_dir = next((d for d in (f'{SPLITS_DATASET}/Chittagonian_AI_Project', SPLITS_DATASET)
                 if os.path.exists(f'{d}/data_manifest.json')), None)
if data_dir is None:      # fallback: look anywhere under /kaggle/input
    _found = sorted(glob.glob('/kaggle/input/**/data_manifest.json', recursive=True))
    assert _found, (f"data_manifest.json not found in {SPLITS_DATASET} (or anywhere under /kaggle/input). "
                    "Add the dataset 'chittagonian-splits' to this notebook: Add Input > Datasets.")
    data_dir = os.path.dirname(_found[0])
    print("[WARN] expected dataset path not found, using:", data_dir)
print("Frozen splits read from:", data_dir)

# -- FIX: fixed seed for Python / NumPy / PyTorch (CPU + CUDA) -------------
SEED = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU found -> Settings > Accelerator > GPU (T4 or P100)"
print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | "
      f"transformers {transformers.__version__}")

# -- FIX: every model must see the SAME frozen test set -------------------
# 00_data_prep.ipynb writes data_manifest.json with an MD5 fingerprint of each
# split. If a split file changed, we stop instead of silently using new data.
def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

manifest_path = f'{data_dir}/data_manifest.json'
assert os.path.exists(manifest_path), "data_manifest.json missing -> run 00_data_prep.ipynb first."
manifest = json.load(open(manifest_path, encoding='utf-8'))
for split in ['train', 'val', 'test']:
    p = f'{data_dir}/{split}_data.csv'
    assert md5(p) == manifest[split]['md5'], \
        f"{split}_data.csv differs from data_manifest.json -> results would not be comparable!"
print(f"[OK] Splits match manifest (test MD5 {manifest['test']['md5'][:10]}...) - same data as every other model.")

def load_split(name):
    # keep_default_na=False: pandas would otherwise turn strings like "NA" into NaN
    df = pd.read_csv(f'{data_dir}/{name}_data.csv', keep_default_na=False)
    df['chittagonian']    = df['chittagonian'].astype(str)
    df['standard_bangla'] = df['standard_bangla'].astype(str)
    return df

test_df              = load_split('test')
chittagonian_texts   = test_df['chittagonian'].tolist()
standard_bangla_refs = test_df['standard_bangla'].tolist()

# -- FIX: ONE decoding setting for zero-shot AND fine-tuned, for all models -
# (old code: num_beams=3 zero-shot vs num_beams=4 fine-tuned)
MAX_LEN   = 64
NUM_BEAMS = 4

def make_gen_config(model, tok, mtype, tgt_lang_id=None):
    """Fresh GenerationConfig so no model keeps hidden decoding defaults
    (e.g. its own length_penalty / no_repeat_ngram_size from the Hub)."""
    mc = model.config
    dec_start = getattr(mc, "decoder_start_token_id", None)
    if mtype == "bert2bert":
        dec_start = tok.cls_token_id
    g = GenerationConfig(
        max_length=MAX_LEN, num_beams=NUM_BEAMS, do_sample=False,
        length_penalty=1.0, early_stopping=False,
        decoder_start_token_id=dec_start,
        bos_token_id=getattr(mc, "bos_token_id", None),
        eos_token_id=tok.sep_token_id if mtype == "bert2bert" else getattr(mc, "eos_token_id", None),
        pad_token_id=tok.pad_token_id,
    )
    if mtype == "nllb":
        g.forced_bos_token_id = tgt_lang_id
    return g

def detok_wordpiece(s):
    # BERT's WordPiece decoder puts a space before punctuation ("খাই ।").
    # Re-attach it so BLEU's word matching is not unfairly penalised.
    return re.sub(r'\s+([।,?!;:.\)])', r'\1', s)

def translate(model, tok, texts, mtype, prefix="", tgt_lang_id=None, batch_size=8):
    model.eval()
    gen_cfg = make_gen_config(model, tok, mtype, tgt_lang_id)
    preds, n_failed = [], 0
    for i in range(0, len(texts), batch_size):
        batch = [prefix + t for t in texts[i:i + batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(device)
        try:
            with torch.no_grad():
                out = model.generate(input_ids=enc["input_ids"],
                                     attention_mask=enc["attention_mask"],
                                     generation_config=gen_cfg)
            decoded = tok.batch_decode(out, skip_special_tokens=True)
        except Exception as e:
            print(f"  [WARN] Batch {i//batch_size}: {e}")
            decoded, n_failed = [""] * len(batch), n_failed + len(batch)
        if mtype == "bert2bert":
            decoded = [detok_wordpiece(d) for d in decoded]
        preds.extend(d.strip() for d in decoded)
    # FIX: failed batches are counted and saved, so empty outputs cannot hide
    print(f"  Generated {len(preds)} | failed: {n_failed}")
    return preds, n_failed

# -- Metrics (identical for every model) ----------------------------------
_bleu   = BLEU()   # sacreBLEU defaults (13a tokenizer) -> signature saved for the paper
_chrf   = CHRF()   # chrF: char 6-grams, beta=2 (not chrF++)
rouge_m = evaluate.load("rouge")
bert_m  = evaluate.load("bertscore")

def score(preds, refs):
    bleu = _bleu.corpus_score(preds, [refs]).score
    chrf = _chrf.corpus_score(preds, [refs]).score
    rl   = rouge_m.compute(predictions=preds, references=refs,
                           tokenizer=lambda x: x.split())['rougeL']   # word-level, Bengali-safe
    try:
        bs   = bert_m.compute(predictions=preds, references=refs, lang="bn",
                              model_type="xlm-roberta-base", batch_size=32)
        bert, bs_hash = round(float(np.mean(bs['f1'])) * 100, 2), bs.get('hashcode', '')
    except Exception as e:
        print(f"  [WARN] BERTScore: {e}"); bert, bs_hash = float('nan'), ''
    row = {"BLEU": round(bleu, 2), "ROUGE-L": round(rl * 100, 2),
           "chrF": round(chrf, 2), "BERTScore (F1)": bert}
    sig = {"bleu_signature": str(_bleu.get_signature()),
           "chrf_signature": str(_chrf.get_signature()),
           "bertscore_hash": bs_hash}
    return row, sig

def free(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

SAMPLE_IDXS = random.Random(SEED).sample(range(len(chittagonian_texts)), 5)  # same 5 examples everywhere
print(f"Test sentences: {len(chittagonian_texts)} | decoding: beams={NUM_BEAMS}, max_len={MAX_LEN}")


In [ ]:
# -- CELL 4: Model / Tokenizer Loading (all architectures) ---------------
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          BertTokenizer, EncoderDecoderModel)

def _load(cls_fn, src, half=False, **kw):
    # transformers >=4.56 renamed torch_dtype -> dtype; try the new name first.
    if half:
        try:    return cls_fn(src, dtype=torch.float16, **kw)
        except TypeError: return cls_fn(src, torch_dtype=torch.float16, **kw)
    return cls_fn(src, **kw)

def configure_bert2bert(model, tok):
    # BERT has no "start of decoding" token, so we tell the model to use [CLS]
    # to start and [SEP] to stop. Set on BOTH config and generation_config.
    model.config.decoder_start_token_id = tok.cls_token_id
    model.config.eos_token_id           = tok.sep_token_id
    model.config.pad_token_id           = tok.pad_token_id
    model.config.vocab_size             = model.config.decoder.vocab_size
    for k in ("decoder_start_token_id", "eos_token_id", "pad_token_id"):
        setattr(model.generation_config, k, getattr(model.config, k))
    # FIX (sanity): tokenizer and model vocab must agree, else ids point to wrong rows
    assert len(tok) <= model.config.decoder.vocab_size, \
        f"Tokenizer has {len(tok)} tokens but decoder vocab is {model.config.decoder.vocab_size}"

def load_model(source, finetuned, for_inference):
    """source = Hub id (pretrained) or Drive folder (fine-tuned)."""
    mtype, tgt_lang_id = CFG["type"], None
    if mtype == "seq2seq":
        tok   = AutoTokenizer.from_pretrained(source)
        model = _load(AutoModelForSeq2SeqLM.from_pretrained, source)
    elif mtype == "nllb":
        tok   = AutoTokenizer.from_pretrained(source, src_lang=CFG["src_lang"], tgt_lang=CFG["tgt_lang"])
        # fp16 only for inference (T4 memory); training keeps fp32 weights + AMP
        model = _load(AutoModelForSeq2SeqLM.from_pretrained, source, half=for_inference)
        try:    tgt_lang_id = tok.lang_code_to_id[CFG["tgt_lang"]]
        except: tgt_lang_id = tok.convert_tokens_to_ids(CFG["tgt_lang"])
        # FIX: transformers v5 refuses to save generation settings inside model.config
        model.generation_config.forced_bos_token_id = tgt_lang_id
    elif mtype == "bert2bert":
        if finetuned:
            tok   = BertTokenizer.from_pretrained(source)       # saved settings are reloaded
            model = EncoderDecoderModel.from_pretrained(source)
        else:
            tok   = BertTokenizer.from_pretrained(source, **TOK_KW)
            model = EncoderDecoderModel.from_encoder_decoder_pretrained(
                        source, source, tie_encoder_decoder=CFG.get("tie_weights", False))
        configure_bert2bert(model, tok)
    else:
        raise ValueError(mtype)
    return tok, model.to(device), tgt_lang_id

TOK_KW = {}   # filled by the tokenizer check (Cell 5) for BERT2BERT


In [ ]:
# -- CELL 5: Tokenizer Sanity Check (reported for every model) -----------
# Why: if a tokenizer cannot represent Bengali properly (unknown tokens, or
# vowel signs like ি ু ্ stripped by "strip_accents"), no amount of training helps.
# We also measure how many reference sentences are longer than MAX_LEN tokens.
import unicodedata
def nfc(s): return unicodedata.normalize("NFC", s)

def tokenizer_report(tok, texts, refs):
    n_tok = n_unk = exact = too_long = 0
    for t in texts[:500]:
        ids = tok(t, add_special_tokens=False)['input_ids']
        n_tok += len(ids); n_unk += sum(i == tok.unk_token_id for i in ids)
        dec = tok.decode(ids, skip_special_tokens=True)
        exact += nfc(dec).replace(" ", "") == nfc(t).replace(" ", "")
    for r in refs:
        too_long += len(tok(r)['input_ids']) > MAX_LEN
    return {"unk_rate_%": round(100 * n_unk / max(n_tok, 1), 2),
            "roundtrip_exact_%": round(100 * exact / min(len(texts), 500), 1),
            "refs_over_max_len_%": round(100 * too_long / len(refs), 2)}

if CFG["type"] == "bert2bert":
    # Try the Hub default and a "keep every character" setting; keep the better one.
    candidates = {"hub_default": {},
                  "keep_marks":  {"do_lower_case": False, "strip_accents": False}}
    reports = {}
    for name, kw in candidates.items():
        t = BertTokenizer.from_pretrained(CFG["model_id"], **kw)
        reports[name] = tokenizer_report(t, chittagonian_texts, standard_bangla_refs)
        print(f"  {name:<12} {reports[name]}")
    best = max(reports, key=lambda n: (reports[n]["roundtrip_exact_%"], -reports[n]["unk_rate_%"]))
    TOK_KW = candidates[best]
    TOK_REPORT = reports[best]
    print(f"-> Using tokenizer setting: {best} {TOK_KW}")
else:
    t = (AutoTokenizer.from_pretrained(CFG["model_id"], src_lang=CFG["src_lang"])
         if CFG["type"] == "nllb" else AutoTokenizer.from_pretrained(CFG["model_id"]))
    TOK_REPORT = tokenizer_report(t, chittagonian_texts, standard_bangla_refs)
    print(f"  {TOK_REPORT}")
if TOK_REPORT["unk_rate_%"] > 1 or TOK_REPORT["roundtrip_exact_%"] < 90:
    print("[WARN] Tokenizer loses information on this data - mention this in the paper.")


In [ ]:
# -- CELL 6: Zero-Shot Evaluation (before any training) ------------------
print(f"-- Zero-Shot: {MODEL_NAME} " + "-" * 40)
tok, model, tgt_lang_id = load_model(CFG["model_id"], finetuned=False, for_inference=True)
zs_preds, zs_failed = translate(model, tok, chittagonian_texts, CFG["type"],
                                prefix=CFG.get("task_prefix", ""), tgt_lang_id=tgt_lang_id,
                                batch_size=CFG["eval_bs"])
free(model, tok); del model, tok; gc.collect(); torch.cuda.empty_cache()

zs_row, zs_sig = score(zs_preds, standard_bangla_refs)
zs_row = {"Model": MODEL_NAME, "State": "Zero-Shot", **zs_row,
          "n_test": len(zs_preds), "n_failed": zs_failed}
pd.DataFrame([zs_row]).to_csv(f'{results_dir}/baseline_scores_{MODEL_NAME}.csv', index=False)
pd.DataFrame({"chittagonian": chittagonian_texts, "reference": standard_bangla_refs,
              "prediction": zs_preds}).to_csv(f'{results_dir}/preds_zeroshot_{MODEL_NAME}.csv', index=False)
display(pd.DataFrame([zs_row]))


In [ ]:
# -- CELL 7: Fine-Tuning -------------------------------------------------
import math
from datasets import Dataset
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainingArguments,
                          Seq2SeqTrainer, EarlyStoppingCallback)
set_seed(SEED)
save_path = f'{project_dir}/finetuned_{MODEL_NAME}'
ckpt_dir  = f'/kaggle/working/ckpt_{MODEL_NAME}'          # local disk: per-epoch checkpoints
print(f"TRAINING {MODEL_NAME} | {HP}")

tok, model, tgt_lang_id = load_model(CFG["model_id"], finetuned=False, for_inference=False)
train_ds = Dataset.from_pandas(load_split('train'))
val_ds   = Dataset.from_pandas(load_split('val'))

def preprocess(examples):
    inputs = [CFG.get("task_prefix", "") + t for t in examples['chittagonian']]
    if CFG["type"] == "bert2bert":
        enc = tok(inputs, max_length=MAX_LEN, truncation=True, padding='max_length')
        tgt = tok(examples['standard_bangla'], max_length=MAX_LEN + 1,
                  truncation=True, padding='max_length')['input_ids']
        # FIX: drop the leading [CLS] from labels. The model already feeds [CLS]
        # as the first decoder input, so labels must start at the first real word.
        enc['labels'] = [[(t if t != tok.pad_token_id else -100) for t in seq[1:]] for seq in tgt]
        return enc
    out = tok(inputs, text_target=examples['standard_bangla'],
              max_length=MAX_LEN, truncation=True, padding='max_length')
    out['labels'] = [[(l if l != tok.pad_token_id else -100) for l in lab] for lab in out['labels']]
    return out

tok_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(preprocess,   batched=True, remove_columns=val_ds.column_names)
if "token_type_ids" in tok_train.column_names:
    tok_train = tok_train.remove_columns("token_type_ids")
    tok_val   = tok_val.remove_columns("token_type_ids")
collator = DataCollatorForSeq2Seq(tokenizer=tok, model=model, padding=True)

steps_per_epoch = math.ceil(len(tok_train) / (HP["bs"] * HP["ga"]))
warmup_steps    = int(HP["warmup_ratio"] * steps_per_epoch * HP["epochs"])

args = Seq2SeqTrainingArguments(
    output_dir                    = ckpt_dir,
    eval_strategy                 = "epoch",
    save_strategy                 = "epoch",          # needed to keep the best epoch
    logging_strategy              = "epoch",
    learning_rate                 = HP["lr"],
    lr_scheduler_type             = "linear",
    warmup_steps                  = warmup_steps,
    per_device_train_batch_size   = HP["bs"],
    per_device_eval_batch_size    = HP["bs"],
    gradient_accumulation_steps   = HP["ga"],
    weight_decay                  = 0.01,
    max_grad_norm                 = 1.0,
    num_train_epochs              = HP["epochs"],
    fp16                          = HP["fp16"],
    gradient_checkpointing        = HP["ckpt"],
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    # FIX: keep the epoch with the LOWEST validation loss (not just the last one)
    load_best_model_at_end        = True,
    metric_for_best_model         = "eval_loss",
    greater_is_better             = False,
    save_total_limit              = 2,
    save_only_model               = True,             # no optimizer state -> less disk
    predict_with_generate         = False,            # val = loss only -> much faster
    seed                          = SEED,
    data_seed                     = SEED,
    report_to                     = "none",
)
trainer = Seq2SeqTrainer(
    model=model, args=args, train_dataset=tok_train, eval_dataset=tok_val,
    processing_class=tok, data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=HP["patience"])],
)

t0 = time.time()
train_out = trainer.train()
train_minutes = round((time.time() - t0) / 60, 1)

trainer.save_model(save_path)        # this is the BEST epoch (load_best_model_at_end)
tok.save_pretrained(save_path)
model.generation_config.save_pretrained(save_path)

logs = trainer.state.log_history
hist = {"train_loss": [l['loss'] for l in logs if 'loss' in l and 'eval_loss' not in l],
        "eval_loss":  [l['eval_loss'] for l in logs if 'eval_loss' in l]}
best_epoch = int(np.argmin(hist["eval_loss"])) + 1
hist.update({"best_epoch": best_epoch, "best_eval_loss": float(min(hist["eval_loss"])),
             "epochs_run": len(hist["eval_loss"]), "train_minutes": train_minutes})
save_json(hist, f'{results_dir}/loss_hist_{MODEL_NAME}.json')
print(f"Saved best model (epoch {best_epoch}, val loss {hist['best_eval_loss']:.4f}) -> {save_path}")

# Stop early if training clearly failed (loss still ~ random guessing)
vocab = model.config.decoder.vocab_size if CFG["type"] == "bert2bert" else model.config.vocab_size
if hist["train_loss"][-1] > 0.8 * math.log(vocab):
    print(f"[ERROR] Final train loss {hist['train_loss'][-1]:.2f} ~ ln(vocab)={math.log(vocab):.2f} "
          "-> the model did not learn. Do NOT report these numbers; check LR / tokenizer.")

free(trainer, model); del trainer, model; gc.collect(); torch.cuda.empty_cache()
shutil.rmtree(ckpt_dir, ignore_errors=True)


In [ ]:
# -- CELL 8: Training & Validation Loss Curve ----------------------------
import matplotlib.pyplot as plt, seaborn as sns
hist   = json.load(open(f'{results_dir}/loss_hist_{MODEL_NAME}.json'))
color  = CFG["color"]
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 15})
ep_tr = range(1, len(hist["train_loss"]) + 1)
ep_ev = range(1, len(hist["eval_loss"]) + 1)
plt.figure(figsize=(10, 6))
plt.plot(ep_tr, hist["train_loss"], marker="o", lw=2.5, color=color, label="Training Loss")
plt.plot(ep_ev, hist["eval_loss"],  marker="s", lw=2.5, color=color, ls="--", alpha=0.75,
         label="Validation Loss")
plt.axvline(hist["best_epoch"], color="grey", ls=":", lw=1.5,
            label=f"Best epoch ({hist['best_epoch']})")
plt.title(f"{MODEL_NAME}: Training vs Validation Loss", fontweight="bold", pad=15)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.xticks(list(ep_ev)); plt.legend(); plt.tight_layout()
path = f'{results_dir}/loss_{MODEL_NAME}.png'
plt.savefig(path, dpi=300, bbox_inches="tight"); plt.show(); plt.close()
print(f"Saved: {path}")


In [ ]:
# -- CELL 9: Fine-Tuned Evaluation (same test set, same decoding) --------
print(f"-- Fine-Tuned: {MODEL_NAME} " + "-" * 40)
save_path = f'{project_dir}/finetuned_{MODEL_NAME}'
tok, model, tgt_lang_id = load_model(save_path, finetuned=True, for_inference=True)
ft_preds, ft_failed = translate(model, tok, chittagonian_texts, CFG["type"],
                                prefix=CFG.get("task_prefix", ""), tgt_lang_id=tgt_lang_id,
                                batch_size=CFG["eval_bs"])
free(model, tok); del model, tok; gc.collect(); torch.cuda.empty_cache()

ft_row, ft_sig = score(ft_preds, standard_bangla_refs)
ft_row = {"Model": MODEL_NAME, "State": "Fine-Tuned", **ft_row,
          "n_test": len(ft_preds), "n_failed": ft_failed}
pd.DataFrame([ft_row]).to_csv(f'{results_dir}/finetuned_scores_{MODEL_NAME}.csv', index=False)
pd.DataFrame({"chittagonian": chittagonian_texts, "reference": standard_bangla_refs,
              "prediction": ft_preds}).to_csv(f'{results_dir}/preds_finetuned_{MODEL_NAME}.csv', index=False)
display(pd.DataFrame([zs_row, ft_row]) if 'zs_row' in globals() else pd.DataFrame([ft_row]))

print("\n-- Sample Translations " + "-" * 46)
zs_map = (pd.read_csv(f'{results_dir}/preds_zeroshot_{MODEL_NAME}.csv', keep_default_na=False)['prediction'].tolist()
          if os.path.exists(f'{results_dir}/preds_zeroshot_{MODEL_NAME}.csv') else None)
for idx in SAMPLE_IDXS:
    print(f"\n  Chittagonian : {chittagonian_texts[idx]}")
    print(f"  True Standard: {standard_bangla_refs[idx]}")
    if zs_map: print(f"  Zero-Shot    : {zs_map[idx]}")
    print(f"  Fine-Tuned   : {ft_preds[idx]}")
print("-" * 68)


In [ ]:
# -- CELL 10: Save Run Record (for the paper's reproducibility section) --
import importlib.metadata as md
def ver(p):
    try: return md.version(p)
    except Exception: return "n/a"
hist = json.load(open(f'{results_dir}/loss_hist_{MODEL_NAME}.json'))
run = {
    "model_name": MODEL_NAME, "config": CFG, "hyperparameters": HP,
    "decoding": {"num_beams": NUM_BEAMS, "max_length": MAX_LEN, "do_sample": False, "length_penalty": 1.0},
    "seed": SEED, "tokenizer_kwargs": TOK_KW, "tokenizer_report": TOK_REPORT,
    "data": {k: manifest[k] for k in ["train", "val", "test"]},
    "best_epoch": hist["best_epoch"], "epochs_run": hist["epochs_run"],
    "train_minutes": hist["train_minutes"],
    "metric_signatures": ft_sig,
    "versions": {p: ver(p) for p in ["torch", "transformers", "datasets", "evaluate",
                                     "sacrebleu", "rouge_score", "bert_score", "accelerate"]},
    "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(),
}
save_json(run, f'{results_dir}/run_config_{MODEL_NAME}.json')
print(json.dumps(run, indent=2, ensure_ascii=False)[:2500])
print(f"\n[OK] {MODEL_NAME} done. Files are in {results_dir}. Next: Cell 11 packs them into a zip for download.")


In [ ]:
# -- CELL 11: Package results for download --------------------------------
# Small files only (scores, predictions, loss curve, run record). The big model folder stays in the Output tab.
import zipfile
zpath = f'/kaggle/working/results_{MODEL_NAME}.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(results_dir)):
        if f.endswith(('.csv', '.json', '.png')) and MODEL_NAME in f:
            z.write(f'{results_dir}/{f}', f)
print(f"[OK] {zpath}  ({os.path.getsize(zpath)/1024:.0f} KB)")
with zipfile.ZipFile(zpath) as z: print("\n".join("  - " + n for n in z.namelist()))